In [1]:
documents = {
    'doc1': "The quick brown fox jumps over the lazy dog. The dog is lazy.",
    'doc2': "A brown cat sat on the mat. The cat is quick.",
    'doc3': "Foxes are clever. Dogs are loyal. Cats are independent.",
    'doc4': "Quick dogs and lazy cats are good pets."
}

In [2]:
print("Sample Documents:")
for doc_id, content in documents.items():
    print(f"{doc_id}: {content}")

Sample Documents:
doc1: The quick brown fox jumps over the lazy dog. The dog is lazy.
doc2: A brown cat sat on the mat. The cat is quick.
doc3: Foxes are clever. Dogs are loyal. Cats are independent.
doc4: Quick dogs and lazy cats are good pets.


In [4]:
!pip install nltk

  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 4.8 MB/s  0:00:00 eta 0:00:01
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [nltk]2/3 [nltk]


In [5]:
import re
import nltk
from nltk.corpus import stopwords

# Download NLTK stop words if not already downloaded
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

/opt/anaconda3/envs/agent/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to /Users/riya/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [6]:
def preprocess_text(text):
    # Tokenize and convert to lowercase
    tokens = re.findall(r'\b\w+\b', text.lower())
    # Remove stop words
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return filtered_tokens

In [7]:
processed_documents = {}
for doc_id, content in documents.items():
    processed_documents[doc_id] = preprocess_text(content)

print("Processed Documents:")
for doc_id, tokens in processed_documents.items():
    print(f"{doc_id}: {tokens}")

Processed Documents:
doc1: ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog', 'dog', 'lazy']
doc2: ['brown', 'cat', 'sat', 'mat', 'cat', 'quick']
doc3: ['foxes', 'clever', 'dogs', 'loyal', 'cats', 'independent']
doc4: ['quick', 'dogs', 'lazy', 'cats', 'good', 'pets']


In [8]:
"""### Step 3: Build the Inverted Index

An inverted index maps each unique term to a list of documents in which it appears. For our basic implementation, we'll store a dictionary where keys are terms and values are sets of document IDs.

Using sets ensures that each document ID is stored only once for a given term, and it provides efficient lookup and union operations for query processing.
"""

inverted_index = {}

for doc_id, tokens in processed_documents.items():
    for token in tokens:
        if token not in inverted_index:
            inverted_index[token] = set()
        inverted_index[token].add(doc_id)

print("Inverted Index:")
for term, doc_ids in inverted_index.items():
    print(f"'{term}': {sorted(list(doc_ids))}")

Inverted Index:
'quick': ['doc1', 'doc2', 'doc4']
'brown': ['doc1', 'doc2']
'fox': ['doc1']
'jumps': ['doc1']
'lazy': ['doc1', 'doc4']
'dog': ['doc1']
'cat': ['doc2']
'sat': ['doc2']
'mat': ['doc2']
'foxes': ['doc3']
'clever': ['doc3']
'dogs': ['doc3', 'doc4']
'loyal': ['doc3']
'cats': ['doc3', 'doc4']
'independent': ['doc3']
'good': ['doc4']
'pets': ['doc4']


In [9]:
"""### Step 4: Implement Document Retrieval

Now that we have our inverted index, we can use it to retrieve documents based on a user's query.

The retrieval process will involve:
1.  **Preprocessing the query:** The query needs to be processed in the same way as the documents (tokenization, lowercasing, stop word removal).
2.  **Looking up terms in the inverted index:** For each term in the preprocessed query, we'll find the set of documents it appears in.
3.  **Combining results:** For a multi-term query, we typically take the intersection of the document sets (for an 'AND' query) to find documents that contain *all* query terms. For an 'OR' query, we would take the union.
"""

def search_inverted_index(query, inverted_index, documents):
    processed_query = preprocess_text(query)
    print(f"Processed query: {processed_query}")

    if not processed_query:
        return []

    # Start with the document set for the first term
    # If the term is not in the index, start with an empty set
    results = inverted_index.get(processed_query[0], set()).copy()

    # Intersect with results for subsequent terms (AND logic)
    for term in processed_query[1:]:
        if not results: # Optimization: if results are already empty, no need to check further
            break
        term_docs = inverted_index.get(term, set())
        results = results.intersection(term_docs)

    if not results:
        return []
    else:
        # Return the actual document content for found doc_ids
        found_docs_content = {doc_id: documents[doc_id] for doc_id in sorted(list(results))}
        return found_docs_content


In [10]:
# Test the search function
query1 = "quick brown fox"
retrieved_docs1 = search_inverted_index(query1, inverted_index, documents)
print(f"\nDocuments for query '{query1}':")
if retrieved_docs1:
    for doc_id, content in retrieved_docs1.items():
        print(f"  {doc_id}: {content}")
else:
    print("  No documents found.")

Processed query: ['quick', 'brown', 'fox']

Documents for query 'quick brown fox':
  doc1: The quick brown fox jumps over the lazy dog. The dog is lazy.


In [11]:
query2 = "lazy cats"
retrieved_docs2 = search_inverted_index(query2, inverted_index, documents)
print(f"\nDocuments for query '{query2}':")
if retrieved_docs2:
    for doc_id, content in retrieved_docs2.items():
        print(f"  {doc_id}: {content}")
else:
    print("  No documents found.")


Processed query: ['lazy', 'cats']

Documents for query 'lazy cats':
  doc4: Quick dogs and lazy cats are good pets.


In [12]:
query3 = "elephant"
retrieved_docs3 = search_inverted_index(query3, inverted_index, documents)
print(f"\nDocuments for query '{query3}':")
if retrieved_docs3:
    for doc_id, content in retrieved_docs3.items():
        print(f"  {doc_id}: {content}")
else:
    print("  No documents found.")

Processed query: ['elephant']

Documents for query 'elephant':
  No documents found.
